# Tutorial 2: Phase analysis with tree search
Dara is equipped with a parallelized tree search algorithm to identify possible phases
present in a given XRD pattern.

In [1]:
# %pip install ipywidgets nbformat

In [2]:
from pathlib import Path

from dara import search_phases

In [3]:
pattern_path = r"D:\AMIT_WORK\projects\xrd_pipeline\new_data\OCX 1401026_exported.xy"
chemical_system = "Li-Ni-Mn-Co-O-H"

In [4]:
import re

def strip_xy_header(path: str) -> str:
    """Return a path to a header-stripped version of the .xy file.

    COMMANDER-exported .xy files have a metadata line at the top that
    dara cannot parse. This strips any lines that don't begin with a number
    and writes a clean copy next to the original (suffix _clean.xy).
    """
    p = Path(path)
    lines = p.read_text(encoding="utf-8").splitlines()
    data_lines = [l for l in lines if re.match(r"^\s*[\d.\-]", l)]
    clean_path = p.parent / (p.stem + "_clean.xy")
    clean_path.write_text("\n".join(data_lines), encoding="utf-8")
    return str(clean_path)

pattern_path = strip_xy_header(pattern_path)
print(f"Using cleaned pattern file: {pattern_path}")

Using cleaned pattern file: D:\AMIT_WORK\projects\xrd_pipeline\new_data\OCX 1401026_exported_clean.xy


## Step 1: Prepare reference phases

In [5]:
import time
import tempfile
import requests
from pathlib import Path
from dara.cif import Cif
from dara.structure_db import CODDatabase

COD_URL = "https://www.crystallography.net/cod/{cod_id}.cif"
_COD_SESSION = requests.Session()
_COD_SESSION.headers.update({
    "User-Agent": "Mozilla/5.0 (research use; contact: your@email.com)"
})

def _download_cod_windows_fix(cod_id: str) -> Cif:
    max_retries = 5
    for attempt in range(max_retries):
        try:
            # Rate-limit: base delay + increasing penalty on each retry
            time.sleep(0.5 + attempt * 1.5)
            url = COD_URL.format(cod_id=cod_id)
            response = _COD_SESSION.get(url, timeout=30)
            response.raise_for_status()
            with tempfile.NamedTemporaryFile(mode="w+b", suffix=".cif", delete=False) as f:
                temp_path = f.name
                f.write(response.content)
            try:
                cif = Cif.from_file(temp_path)
            finally:
                Path(temp_path).unlink(missing_ok=True)
            cif.filename = str(cod_id)
            return cif
        except Exception as e:
            if attempt < max_retries - 1:
                wait = 2 ** attempt  # exponential backoff: 1, 2, 4, 8 s
                print(f"  [{cod_id}] attempt {attempt+1} failed ({e}), retrying in {wait}s…")
                time.sleep(wait)
            else:
                print(f"  [{cod_id}] all {max_retries} attempts failed: {e}")
                raise

CODDatabase._download_cod = staticmethod(_download_cod_windows_fix)


In [6]:
from dara.structure_db import CODDatabase

cod_database = CODDatabase()
all_icsd_ids = cod_database.get_cifs_by_chemsys(chemical_system, dest_dir="cifs")

2026-10-05 11:43:51,533 WARNING dara.structure_db Local copy of database not found. Attempting to download structures...


2026-10-05 11:44:25,033 INFO dara.structure_db Saving downloaded CIFs to dara_downloaded_cifs
2026-10-05 11:44:27,533 INFO dara.structure_db Skipping common gas: O2
2026-10-05 11:44:27,533 INFO dara.structure_db Skipping common gas: O2
2026-10-05 11:44:27,538 INFO dara.structure_db Skipping common gas: O2
2026-10-05 11:44:27,538 INFO dara.structure_db Skipping common gas: O2
2026-10-05 11:44:27,538 INFO dara.structure_db Skipping common gas: O2
2026-10-05 11:44:27,538 INFO dara.structure_db Skipping common gas: O2
2026-10-05 11:44:27,538 INFO dara.structure_db Skipping common gas: O2
2026-10-05 11:44:27,538 INFO dara.structure_db Skipping common gas: O2
2026-10-05 11:44:27,538 INFO dara.structure_db Skipping common gas: O2
2026-10-05 11:44:27,538 INFO dara.structure_db Skipping common gas: O2
2026-10-05 11:44:27,538 INFO dara.structure_db Skipping common gas: O2
2026-10-05 11:44:27,538 INFO dara.structure_db Skipping common gas: O2
2026-10-05 11:44:27,538 INFO dara.structure_db Skippin

Since we are using a pre-filterd database (i.e., the COD), the downloaded CIF files will automatically be named according to the
following convention:

```
{composition}_{spacegroup}_(cod|icsd_{id})-{e_hull}.cif
```
Where the `e_hull` is the energy above the convex hull in meV/atom, as determined from
the Materials Project database for the ground-state entry with matching composition and spacegroup.

## Step 2: Search for phases

After preparing the reference CIFs, we can start the phase search on a provided XRD
pattern.

In [7]:
import ray

# Windows fix: stale Ray instances from a previous run cause PermissionError in workers.
# Always do a clean shutdown + restart before searching.
if ray.is_initialized():
    ray.shutdown()
ray.init(runtime_env={"working_dir": None})

cifs_dir = Path("cifs").resolve()

curated_cifs = [
    f for f in cifs_dir.glob("*.cif")
    # if any(p in f.name for p in keep_patterns)
    # and not any(p in f.name for p in exclude_patterns)
]
print(f"Curated set: {len(curated_cifs)} phases (from {len(list(cifs_dir.glob('*.cif')))} total)")
for f in sorted(curated_cifs):
    print(f"  {f.name}")

search_results = search_phases(
    pattern_path=pattern_path,
    phases=curated_cifs,
    wavelength="Cu",
    instrument_profile="Aeris-fds-Pixcel1d-Medipix3",
    express_mode=False,  
)

2026-10-05 11:44:41,819	INFO worker.py:2024 -- Started a local Ray instance.


Curated set: 311 phases (from 311 total)
  Co(HO)2_164_(cod_1548810)-37.cif
  Co0.048Ni3.952H4.24O8_12_(cod_4109372)-None.cif
  Co1.5Ni0.5_194_(cod_1525374)-None.cif
  Co19.5O48_160_(cod_1536059)-None.cif
  Co1H1.99998O2_164_(cod_9006535)-None.cif
  Co1O1.92_164_(cod_1522026)-None.cif
  Co20.96O32_227_(cod_1528447)-None.cif
  Co21.92O32_227_(cod_1528446)-None.cif
  Co3.592O7.184_10_(cod_1529156)-None.cif
  Co3O4_227_(cod_5910031)-0.cif
  Co3O4_227_(cod_9005896)-0.cif
  Co5.312O12_10_(cod_1526826)-None.cif
  Co_139_(cod_4105680)-None.cif
  Co_194_(cod_9012950)-0.cif
  Co_225_(cod_9012949)-0.cif
  CoHO2_166_(cod_9009884)-0.cif
  CoHO2_194_(cod_9009449)-85.cif
  CoO2_164_(cod_1010267)-8.cif
  CoO2_51_(cod_1526822)-None.cif
  CoO3_166_(cod_1525955)-None.cif
  CoO_225_(cod_1541662)-37.cif
  H2O2_92_(cod_1008189)-86.cif
  H39.984O20_81_(cod_1566653)-None.cif
  H4.0000008O2_224_(cod_1566716)-None.cif
  H4O_224_(cod_1566642)-None.cif
  H53.636O28_14_(cod_1566647)-None.cif
  Li0.24Mn16O32_227_(

(raylet) Stack (most recent call first):
(raylet)   File "C:\Users\amitr\AppData\Roaming\Python\Python312\site-packages\ray\_private\worker.py", line 627 in job_logging_config
(raylet)   File "C:\Users\amitr\AppData\Roaming\Python\Python312\site-packages\ray\_private\worker.py", line 2807 in disconnectWindows fatal exception: access violation
(raylet) 
(raylet)   File Windows fatal exception: access violation
(raylet) 
(raylet)   File "C:\Users\amitr\AppData\Roaming\Python\Python312\site-packages\ray\_private\worker.py", line 2807 in disconnect
(raylet)   File "C:\Users\amitr\AppData\Roaming\Python\Python312\site-packages\ray\_private\worker.py", line 2118 in shutdown
(raylet)   File "C:\Users\amitr\AppData\Roaming\Python\Python312\site-packages\ray\_private\worker.py", line 1170 in wrapper
(raylet) 
(raylet) ", line 1170 in wrapper
(raylet) Stack (most recent call first): [repeated 18x across cluster] (Ray deduplicates logs by default. Set RAY_DEDUP_LOGS=0 to disable log deduplication

(_remote_expand_node pid=55860) 2026-10-05 11:48:35,834 INFO dara.search.tree Expanding node 985b8a47-c084-11f1-8c5f-e073e7b8e1ac with current phases [RefinementPhase(path=WindowsPath('D:/AMIT_WORK/projects/xrd_pipeline/cifs/Li2.34Mn2.55O6_166_(cod_4001901)-None.cif'), params={'k1': '0.010000_0.0^0.01', 'b1': '0.005000_0.0^0.005'})], Rwp = 22.58
(_remote_expand_node pid=57340) 2026-10-05 11:48:45,764 INFO dara.search.tree Expanding node 98723fe8-c084-11f1-b54d-e073e7b8e1ac with current phases [RefinementPhase(path=WindowsPath('D:/AMIT_WORK/projects/xrd_pipeline/cifs/Li3.56Mn8O15.36_141_(cod_1009064)-None.cif'), params={'k1': '0.010000_0.0^0.01', 'b1': '0.005000_0.0^0.005'})], Rwp = 42.59 [repeated 8x across cluster]


(raylet)   File "C:\Users\amitr\AppData\Roaming\Python\Python312\site-packages\ray\_private\worker.py", line 2118 in shutdownWindows fatal exception: 
(raylet)   File "C:\Users\amitr\AppData\Roaming\Python\Python312\site-packages\ray\_private\client_mode_hook.py", line 134 in wrapperaccess violationWindows fatal exception: access violation
(raylet) 
(raylet) 
(raylet) 
(raylet) 
(raylet)   File "C:\Users\amitr\AppData\Roaming\Python\Python312\site-packages\ray\_private\worker.py"  File "C:\Users\amitr\AppData\Roaming\Python\Python312\site-packages\ray\_private\worker.py", line 2807 in disconnect
(raylet)   File "C:\Users\amitr\AppData\Roaming\Python\Python312\site-packages\ray\_private\worker.py", line 1170, line 2807 in disconnect
(raylet)  in wrapper
(raylet)   File "C:\Users\amitr\AppData\Roaming\Python\Python312\site-packages\ray\_private\worker.py", line 627 in job_logging_configWindows fatal exception: 
(raylet)   File access violation
(raylet) 
(raylet) "Stack (most recent call 

(_remote_expand_node pid=57420) 2026-10-05 11:53:36,160 INFO dara.search.tree Expanding node a16b567d-c084-11f1-bc34-e073e7b8e1ac with current phases [RefinementPhase(path=WindowsPath('D:/AMIT_WORK/projects/xrd_pipeline/cifs/Li2MnO3_12_(cod_4001900)-0.cif'), params={'k1': '0.010000_0.0^0.01', 'b1': '0.005000_0.0^0.005'}), RefinementPhase(path=WindowsPath('D:/AMIT_WORK/projects/xrd_pipeline/cifs/H4O_224_(cod_1566642)-None.cif'), params={'k1': '0.010000_0.0^0.01', 'b1': '0.005000_0.0^0.005'})], Rwp = 32.74 [repeated 7x across cluster]
(_remote_expand_node pid=55992) 2026-10-05 11:53:52,015 INFO dara.search.tree Expanding node cb5d4c9a-c084-11f1-a2e5-e073e7b8e1ac with current phases [RefinementPhase(path=WindowsPath('D:/AMIT_WORK/projects/xrd_pipeline/cifs/Li3.56Mn8O15.36_141_(cod_1009064)-None.cif'), params={'k1': '0.010000_0.0^0.01', 'b1': '0.005000_0.0^0.005'}), RefinementPhase(path=WindowsPath('D:/AMIT_WORK/projects/xrd_pipeline/cifs/Li1.998Mn2.002O4_225_(cod_1514037)-None.cif'), para

... (3 outputs left out)

(_remote_expand_node pid=55992) 2026-10-05 11:55:16,594 INFO dara.search.tree Expanding node 7f526221-c085-11f1-9396-e073e7b8e1ac with current phases [RefinementPhase(path=WindowsPath('D:/AMIT_WORK/projects/xrd_pipeline/cifs/Li1.276Mn4.05144O12_12_(cod_4001282)-None.cif'), params={'k1': '0.010000_0.0^0.01', 'b1': '0.005000_0.0^0.005'}), RefinementPhase(path=WindowsPath('D:/AMIT_WORK/projects/xrd_pipeline/cifs/Ni2H_164_(cod_1530159)-25.cif'), params={'k1': '0.010000_0.0^0.01', 'b1': '0.005000_0.0^0.005'})], Rwp = 41.44


(raylet) Stack (most recent call first): [repeated 26x across cluster]
(raylet)   File "C:\Users\amitr\AppData\Roaming\Python\Python312\site-packages\ray\_private\worker.py", line 627 in job_logging_config [repeated 26x across cluster]
(raylet)   File "C:\Users\amitr\AppData\Roaming\Python\Python312\site-packages\ray\_private\worker.py", line 2807 in disconnect [repeated 26x across cluster]
(raylet)   File "C:\Users\amitr\AppData\Roaming\Python\Python312\site-packages\ray\_private\worker.py", line 2118 in shutdown [repeated 26x across cluster]
(raylet)   File "C:\Users\amitr\AppData\Roaming\Python\Python312\site-packages\ray\_private\client_mode_hook.py", line 134 in wrapper [repeated 52x across cluster]
(raylet) Stack (most recent call first): [repeated 13x across cluster]
(raylet)   File "C:\Users\amitr\AppData\Roaming\Python\Python312\site-packages\ray\_private\worker.py", line 627 in job_logging_config [repeated 13x across cluster]
(raylet)   File "C:\Users\amitr\AppData\Roaming\Py

(_remote_expand_node pid=57744) 2026-10-05 11:56:15,957 INFO dara.search.tree Expanding node aaae171b-c085-11f1-8235-e073e7b8e1ac with current phases [RefinementPhase(path=WindowsPath('D:/AMIT_WORK/projects/xrd_pipeline/cifs/Li1.276Mn4.05144O12_12_(cod_4001282)-None.cif'), params={'k1': '0.010000_0.0^0.01', 'b1': '0.005000_0.0^0.005'}), RefinementPhase(path=WindowsPath('D:/AMIT_WORK/projects/xrd_pipeline/cifs/Ni2H_164_(cod_1530159)-25.cif'), params={'k1': '0.010000_0.0^0.01', 'b1': '0.005000_0.0^0.005'}), RefinementPhase(path=WindowsPath('D:/AMIT_WORK/projects/xrd_pipeline/cifs/Co3O4_227_(cod_9005896)-0.cif'), params={'k1': '0.010000_0.0^0.01', 'b1': '0.005000_0.0^0.005'})], Rwp = 39.25 [repeated 8x across cluster]
(_remote_expand_node pid=62520) 2026-10-05 11:56:47,046 INFO dara.search.tree Expanding node ad6308df-c085-11f1-b7a5-e073e7b8e1ac with current phases [RefinementPhase(path=WindowsPath('D:/AMIT_WORK/projects/xrd_pipeline/cifs/Mn10.312Co13.688O32_227_(cod_2300280)-None.cif'), 

(raylet) Stack (most recent call first): [repeated 5x across cluster]
(raylet)   File "C:\Users\amitr\AppData\Roaming\Python\Python312\site-packages\ray\_private\worker.py", line 627 in job_logging_config [repeated 5x across cluster]
(raylet)   File "C:\Users\amitr\AppData\Roaming\Python\Python312\site-packages\ray\_private\worker.py", line 2807 in disconnect [repeated 5x across cluster]
(raylet)   File "C:\Users\amitr\AppData\Roaming\Python\Python312\site-packages\ray\_private\worker.py", line 2118 in shutdown [repeated 5x across cluster]
(raylet)   File "C:\Users\amitr\AppData\Roaming\Python\Python312\site-packages\ray\_private\client_mode_hook.py", line 134 in wrapper [repeated 10x across cluster]
(raylet) Stack (most recent call first): [repeated 10x across cluster]
(raylet)   File "C:\Users\amitr\AppData\Roaming\Python\Python312\site-packages\ray\_private\worker.py", line 627 in job_logging_config [repeated 10x across cluster]
(raylet)   File "C:\Users\amitr\AppData\Roaming\Python

## Step 3: Result analysis
The returned search result will be a list of `SearchResult` object.

In [8]:
search_results

[SearchResult(refinement_result=RefinementResult(lst_data=LstResult(raw_lst='Rietveld refinement to file(s) OCX 1401026_exported_clean.xy\nBGMN version 4.2.22, 668 measured points, 10 peaks, 15 parameters\nStart: Mon Oct  5 11:47:07 2026; End: Mon Oct  5 11:47:08 2026\n17 iteration steps\n\nRp=7.99%  Rpb=24.15%  R=15.98%  Rwp=12.27% Rexp=20.35%\nDurbin-Watson d=1.03\n1-rho=3.21%\n\nGlobal parameters and GOALs\n****************************\nQLiNiO2166cod153377013=1.00000\nEPS2=0.001192+-0.000048\n\nLocal parameters and GOALs for phase LiNiO2166cod153377013\n******************************************************\nSpacegroupNo=166\nHermannMauguin=R-32/m\nXrayDensity=4.766\nRphase=16.74%\nUNIT=NM\nA=0.287669+-0.000032\nC=1.42395+-0.00030\nk1=0.0100000\nB1=0.00500000\nGEWICHT=0.00610+-0.00010\nGrainSize(1,1,1)=84.1811\nAtomic positions for phase LiNiO2166cod153377013\n---------------------------------------------\n  3     0.0000  0.0000  0.5000     E=(LI(0.9780),NI(0.0220))\n  3     0.0000 

In [9]:
for i in range(len(search_results)):
    print(f"Rwp of solution {i} = {search_results[i].refinement_result.lst_data.rwp} %")

Rwp of solution 0 = 12.27 %


Each `SearchResult` has a `.visualize()` method to visualize the refined pattern and
missing/extra peaks in the solution. If there are no missing or extra peaks, this option
will not appear.

In [10]:
search_results[0].visualize()

You can also view all the alternative phases in one solution from `SearchResult.phases` attribute.

In [16]:
print("Phases found in solution 0:")
for i, phases_ in enumerate(search_results[0].phases):
    print(f"    - Phase {i}: {[phase.path.name for phase in phases_]}")

Phases found in solution 0:
    - Phase 0: ['LiNiO2_166_(cod_1533770)-13.cif', 'Li0.65Ni1.08O2_166_(cod_1535512)-None.cif', 'Li3.0Ni3.006O6_166_(cod_1531441)-None.cif', 'Li2.913Ni3.081O6_166_(cod_1531444)-None.cif', 'Li3.03Mn0.3Ni2.67O6_166_(cod_1520787)-None.cif', 'Li3Mn0.9Ni2.1O6_166_(cod_7222435)-None.cif', 'Li2.814Co0.3186Ni2.8674O6_166_(cod_1529128)-None.cif', 'Li2.46Ni3.06O6_166_(cod_1526280)-None.cif', 'Li2.99995053Mn0.99933044Co0.99933044Ni1.00138858O6_166_(cod_4002443)-None.cif', 'Li3.09Mn0.3Co0.3Ni2.31O6_166_(cod_1520789)-None.cif', 'Li3Mn1.2Ni1.8O6_166_(cod_7222436)-None.cif', 'Li2.88Mn2.37O6_166_(cod_4001904)-None.cif', 'Li3Mn1.5Ni1.497O6_166_(cod_4002582)-None.cif', 'Li2.88Mn2.22O6_166_(cod_4001905)-None.cif', 'Li3.6Mn1.5Co0.6Ni0.3O6_166_(cod_1529154)-None.cif', 'Li0.51Ni1.16O2_166_(cod_1535516)-None.cif', 'Li3Mn0.75Co1.5Ni0.75O6_166_(cod_1529155)-None.cif', 'Li1.26Ni2.04O4_166_(cod_1526282)-None.cif', 'Li2.25Ni3.75O6_166_(cod_1526278)-None.cif', 'Li2.772Ni3.228O6_166_(cod

In [18]:

from pymatgen.io.cif import CifParser
import re

def cif_stem_to_bgmn_key(stem):
    """Map a CIF filename stem to the BGMN phase key.

    BGMN derives phase keys by stripping underscores, parentheses, and
    hyphens from the CIF stem.  E.g.:
      'LiNiO2_166_(cod_1533770)-13'  ->  'LiNiO2166cod153377013'
    """
    return re.sub(r"[_()\-]", "", stem)

def get_literature_params_nm(cif_path):
    """Return (a, b, c) in nm from a CIF file using pymatgen.
    Returns (None, None, None) on failure.
    """
    try:
        parser = CifParser(str(cif_path), occupancy_tolerance=10.0)
        structure = parser.parse_structures(primitive=False)[0]
        lat = structure.lattice
        # pymatgen uses Angstroms; convert to nm (BGMN unit)
        return lat.a / 10, lat.b / 10, lat.c / 10
    except Exception as exc:
        print(f"    [CIF read error] {cif_path.name}: {exc}")
        return None, None, None

print("Refined vs Literature Lattice Parameters (unit: nm)")
print("=" * 70)

for sol_idx, result in enumerate(search_results):
    rwp = result.refinement_result.lst_data.rwp
    print(f"\nSolution {sol_idx}  (Rwp = {rwp:.2f} %)")
    print("-" * 70)

    phases_results = result.refinement_result.lst_data.phases_results
    # The first element of .phases is the winning phase set
    best_phases = result.phases[0]
    # Build lookup: bgmn_key -> RefinementPhase
    key_to_path = {cif_stem_to_bgmn_key(rp.path.stem): rp.path for rp in best_phases}

    for phase_key, pr in phases_results.items():
        cif_path = key_to_path.get(phase_key)
        lit_a, lit_b, lit_c = get_literature_params_nm(cif_path) if cif_path else (None, None, None)

        phase_label = cif_path.name if cif_path else phase_key
        print(f"\n  Phase: {phase_label}")
        print(f"  {'Param':<8} {'Refined (nm)':>14} {'± σ':>10} {'Literature (nm)':>16} {'Δ (%)':>8}")
        print(f"  {'-'*58}")

        param_rows = [
            ("a", pr.a, lit_a),
            ("b", pr.b, lit_b),
            ("c", pr.c, lit_c),
        ]
        for pname, refined_tuple, lit_val in param_rows:
            if refined_tuple is None:
                continue
            ref_val, ref_err = refined_tuple
            lit_str  = f"{lit_val:.6f}" if lit_val is not None else "N/A"
            err_str  = f"± {ref_err:.6f}" if ref_err is not None else ""
            if lit_val is not None:
                delta_str = f"{100 * (ref_val - lit_val) / lit_val:+.3f}%"
            else:
                delta_str = ""
            print(f"  {pname:<8} {ref_val:>14.6f} {err_str:>10} {lit_str:>16} {delta_str:>8}")


Refined vs Literature Lattice Parameters (unit: nm)

Solution 0  (Rwp = 12.27 %)
----------------------------------------------------------------------

  Phase: LiNiO2_166_(cod_1533770)-13
  Param      Refined (nm)        ± σ  Literature (nm)    Δ (%)
  ----------------------------------------------------------
  a              0.287669 ± 0.000032              N/A         
  c              1.423950 ± 0.000300              N/A         
